# Heart Disease Prediction System
## Exploratory Data Analysis and Preprocessing Notebook

Project:
Heart Disease Prediction System using End-to-End Machine Learning Pipeline

Dataset:
Heart Disease Dataset (Kaggle)

Objectives:
- Understand dataset characteristics
- Validate data quality
- Perform exploratory data analysis
- Prepare dataset for machine learning
- Generate reproducible preprocessing workflow
- Produce processed datasets for model training

## Cell 2 (Import Libraries)

python3 -m venv .venv
source .venv/bin/activate

pip install -U numpy pandas matplotlib seaborn scipy scikit-learn joblib jupyter notebook

pip freeze > requirements.txt

pip install -r requirements.txt

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import joblib

## Cell 3 (Configuration)

In [2]:
RANDOM_STATE = 42

TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", None)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## Cell 4 (Directory Configuration)

In [ ]:
RAW_DATA_PATH = "../data/raw/heart.csv"

INTERIM_DIR = "../data/interim"
PROCESSED_DIR = "../data/processed"
REPORT_DIR = "../reports"
ARTIFACT_DIR = "../artifacts"

os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

## Cell 5 (Data Loading)

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)

print("Dataset Shape:", df.shape)
df.head()

## Cell 6 (Initial Exploration)

In [ ]:
df.info()
df.describe().T
df.sample(5)

## Cell 7 (Data Validation)

In [ ]:
expected_columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "target"
]

missing_columns = set(expected_columns) - set(df.columns)
additional_columns = set(df.columns) - set(expected_columns)

validation_report = {
    "dataset_shape": df.shape,
    "missing_columns": list(missing_columns),
    "additional_columns": list(additional_columns),
    "duplicate_columns": int(df.columns.duplicated().sum())
}

validation_report

with open(
    f"{REPORT_DIR}/data_validation.json",
    "w"
) as f:
    json.dump(validation_report, f, indent=4)

## Cell 8 (Missing Value Analysis)

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (
        df.isnull().sum()/len(df)
    )*100
})

missing.sort_values(
    by="missing_count",
    ascending=False
)

missing.to_json(
    f"{REPORT_DIR}/missing_value_report.json",
    indent=4
)

## Cell 9 (Duplicate Analysis)

In [ ]:
duplicates = df.duplicated().sum()

print(f"Duplicate rows: {duplicates}")

duplicate_report = {
    "duplicate_rows": int(duplicates)
}

with open(
    f"{REPORT_DIR}/duplicate_report.json",
    "w"
) as f:
    json.dump(
        duplicate_report,
        f,
        indent=4
    )

df = df.drop_duplicates()

print(df.shape)

## Cell 10 (Univariate Analysis)

In [ ]:
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

for col in numerical_features:
    fig, ax = plt.subplots(
        1,
        2,
        figsize=(14,4)
    )

    sns.histplot(
        df[col],
        kde=True,
        ax=ax[0]
    )

    sns.boxplot(
        x=df[col],
        ax=ax[1]
    )

    plt.show()

# categorial

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal",
    "target"
]

for col in categorical_features:
    plt.figure(figsize=(6,4))
    sns.countplot(
        data=df,
        x=col
    )
    plt.show()